# SMIRK — PhD-Level Model Evaluation
## CVPR 2024 · 3D Facial Expression Reconstruction

**Paper:** https://arxiv.org/abs/2404.04104  
**Code:** https://github.com/georgeretsi/smirk

---

This notebook provides a rigorous, publication-grade evaluation of SMIRK across nine orthogonal axes. Each section is designed to be independently runnable and produces a self-contained artefact (figure, table, or CSV).

| # | Dimension | Primary Metric |
|---|---|---|
| 1 | Landmark Alignment | NME (inter-ocular normalised) |
| 2 | Reconstruction Fidelity | SSIM / PSNR / LPIPS |
| 3 | Expression Space Analysis | Coeff norms, active dims, PCA |
| 4 | Shape Consistency | Identity disentanglement r |
| 5 | Crop Ablation | ΔNME, ΔSSIM |
| 6 | Runtime Benchmark | Latency (ms), FPS, VRAM |
| 7 | Resolution Robustness | NME vs resolution curve |
| 8 | Qualitative Visualisation | Input / FLAME / Overlay figures |
| 9 | Statistical Summary + LaTeX | mean±std table, .tar.gz download |


In [ ]:
# ── E1: Environment & GPU Verification ──────────────────────────────────
import torch, sys, os

assert torch.cuda.is_available(), (
    'No CUDA GPU detected.  Runtime ▸ Change runtime type ▸ GPU (T4).'
)

dev = torch.cuda.get_device_properties(0)
print(f'GPU  : {dev.name}')
print(f'VRAM : {dev.total_memory / 1e9:.2f} GB')
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  Python {sys.version.split()[0]}')


In [ ]:

# ── E2: Install all dependencies ─────────────────────────────────────────
import subprocess, sys, urllib.request, os, importlib, traceback

def sh(cmd, extra_env=None):
    env = {**os.environ, **(extra_env or {})}
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    if r.returncode != 0:
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        raise RuntimeError(f'Command failed: {cmd}')
    return r.stdout

import torch
tv  = torch.__version__.split('+')[0].replace('.', '')
cu  = torch.version.cuda.replace('.', '')
pv  = f'py{sys.version_info.major}{sys.version_info.minor}'
tag = f'{pv}_cu{cu}_pyt{tv}'
wheel_url = f'https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/{tag}/download.html'
print(f'torch={torch.__version__} | {pv} | cu{cu}')

# ── numpy <2.0 — must be pinned FIRST ─────────────────────────────────────
print('Pinning numpy<2.0...')
sh("pip install -q 'numpy<2.0'")

import numpy as _np
if int(_np.__version__.split('.')[0]) >= 2:
    print(f'\n⚠ NumPy {_np.__version__} still in memory — restarting kernel...')
    print('  ➜ After restart, re-run THIS cell (Cell 2) again.')
    import os; os.kill(os.getpid(), 9)

print(f'NumPy {_np.__version__} — OK.')

# ── scikit-image: force-reinstall AFTER numpy downgrade ───────────────────
print('Force-reinstalling scikit-image against numpy 1.x...')
sh("pip install -q --force-reinstall 'scikit-image==0.22.0'")

# ── lpips: --no-deps so pip does NOT touch torch/torchvision/numpy ────────
print('Force-reinstalling lpips (no-deps)...')
sh("pip install -q --force-reinstall --no-deps lpips")

# ── scipy: leave whatever version Colab has — we patch lpips instead ──────
# Root cause: lpips/trainer.py imports `from scipy.ndimage import zoom`.
# Fix: patch lpips/trainer.py to stub out the scipy import entirely.
import pathlib as _pl
_trainer = _pl.Path('/usr/local/lib/python3.12/dist-packages/lpips/trainer.py')
if _trainer.exists():
    _src = _trainer.read_text()
    if 'from scipy.ndimage import zoom' in _src:
        _src = _src.replace(
            'from scipy.ndimage import zoom',
            '# scipy.ndimage.zoom patched out — not needed for evaluation\n'
            'zoom = None'
        )
        _trainer.write_text(_src)
        print('Patched lpips/trainer.py: scipy.ndimage.zoom stubbed out.')
    else:
        print('lpips/trainer.py: scipy import already absent.')
else:
    print('WARNING: lpips/trainer.py not found at expected path.')

# ── cv2 typing stubs patch ────────────────────────────────────────────────
# OpenCV pip wheels don't expose several internal attributes that the stubs
# reference unconditionally. Patch ALL known bad lines in one pass.
#
#   Line ~157: Prim = typing.Union[cv2.gapi.wip.draw.Text, ...]
#              → G-API not compiled into Linux pip wheels
#   Line ~162: LayerId = cv2.dnn.DictValue
#              → DictValue removed from opencv-python 4.8+
import glob as _glob, re as _re
_cv2_typing_candidates = (
    _glob.glob('/usr/local/lib/python3.12/dist-packages/cv2/typing/__init__.py') +
    _glob.glob('/usr/local/lib/python3*/dist-packages/cv2/typing/__init__.py')
)
for _cv2_typing in dict.fromkeys(_cv2_typing_candidates):   # deduplicate
    _ct = _pl.Path(_cv2_typing)
    if not _ct.exists():
        continue
    _ts = _ct.read_text()
    _changed = False

    # Fix 1 — cv2.gapi.wip.draw.* (G-API not in pip wheels)
    _bad_gapi = (
        'Prim = typing.Union[cv2.gapi.wip.draw.Text, '
        'cv2.gapi.wip.draw.Circle, cv2.gapi.wip.draw.Image, '
        'cv2.gapi.wip.draw.Line, cv2.gapi.wip.draw.Rect, '
        'cv2.gapi.wip.draw.Mosaic, cv2.gapi.wip.draw.Poly]'
    )
    if _bad_gapi in _ts:
        _ts = _ts.replace(
            _bad_gapi,
            '# cv2.gapi.wip.draw patched out — G-API not available in pip wheels\n'
            'Prim = object'
        )
        _changed = True
        print(f'  [cv2 stub] Patched Prim/gapi line in {_cv2_typing}')

    # Fix 2 — cv2.dnn.DictValue (removed in recent opencv builds)
    if 'LayerId = cv2.dnn.DictValue' in _ts:
        _ts = _ts.replace(
            'LayerId = cv2.dnn.DictValue',
            '# cv2.dnn.DictValue patched out — not present in pip wheels\n'
            'LayerId = object'
        )
        _changed = True
        print(f'  [cv2 stub] Patched LayerId/DictValue line in {_cv2_typing}')

    # Fix 3 — catch-all: any remaining bare cv2.dnn.* or cv2.gapi.* assignments
    # that would raise AttributeError at import time
    for _bad_pattern in _re.findall(
        r'^(\w+\s*=\s*cv2\.(?:dnn|gapi)\.\S+)', _ts, _re.MULTILINE
    ):
        if '# patched' not in _bad_pattern:
            _replacement = f'# patched out — attribute not available\n# {_bad_pattern}'
            _ts = _ts.replace(_bad_pattern, _replacement, 1)
            _changed = True
            print(f'  [cv2 stub] Patched additional line: {_bad_pattern.strip()!r}')

    if _changed:
        _ct.write_text(_ts)
        print(f'cv2 typing stubs updated: {_cv2_typing}')
    else:
        print(f'cv2 typing stubs: all bad references already absent/patched ({_cv2_typing})')

# ── Pillow: force-reinstall to avoid ABI issues after numpy downgrade ─────
print('Force-reinstalling Pillow...')
sh("pip install -q --force-reinstall 'Pillow>=9.5.0,<11.0'")

# ── inspect patch for Python 3.11+ (chumpy uses removed getargspec) ───────
import inspect
if not hasattr(inspect, 'getargspec'):
    inspect.getargspec = inspect.getfullargspec

# ── iopath / fvcore ────────────────────────────────────────────────────────
sh('pip install -q iopath fvcore')

# ── pytorch3d ─────────────────────────────────────────────────────────────
try:
    urllib.request.urlopen(wheel_url)
    print('Prebuilt pytorch3d wheel found — installing...')
    sh(f'pip install -q pytorch3d -f "{wheel_url}"')
except Exception:
    print('✗ No prebuilt wheel — compiling pytorch3d from source (~10-15 min)...')
    sh('pip install -q "git+https://github.com/facebookresearch/pytorch3d.git@stable"',
       extra_env={'MAX_JOBS': '4'})

# ── remaining packages ─────────────────────────────────────────────────────
sh('pip install -q '
   'mediapipe>=0.10.13 timm==0.9.16 '
   'albumentations==1.3.0 omegaconf==2.3.0 chumpy gdown '
   'seaborn matplotlib pandas')

# ── Flush ALL stale C-extension submodule entries from sys.modules ────────
_prefixes = ('lpips', 'scipy', 'skimage', 'PIL', 'cv2')
_evicted = [k for k in list(sys.modules) if k.split('.')[0] in _prefixes]
for _k in _evicted:
    sys.modules.pop(_k, None)
print(f'Evicted {len(_evicted)} stale module cache entries.')

# ── verify ────────────────────────────────────────────────────────────────
_all_ok = True
for pkg in ['pytorch3d', 'mediapipe', 'lpips', 'seaborn', 'scipy', 'skimage']:
    try:
        importlib.import_module(pkg)
        print(f'  ✔ {pkg}')
    except Exception as e:
        _all_ok = False
        print(f'  ✗ {pkg} MISSING  ({e})')
        traceback.print_exc()

if _all_ok:
    print('\n✓ All dependencies ready.')
else:
    print('\n⚠ Some packages failed — see tracebacks above.')


In [ ]:
# ── E3: Clone SMIRK, download checkpoint, upload FLAME ───────────────────
import os, subprocess, pathlib, zipfile

ROOT = pathlib.Path('/content/smirk')
if not ROOT.exists():
    subprocess.run('git clone --depth 1 https://github.com/georgeretsi/smirk /content/smirk',
                   shell=True, check=True)
os.chdir(ROOT)

# SMIRK checkpoint
CKPT = ROOT / 'pretrained_models' / 'SMIRK_em1.pt'
CKPT.parent.mkdir(parents=True, exist_ok=True)
if not CKPT.exists():
    subprocess.run('gdown "https://drive.google.com/uc?id=1T65uEd9dVLHgVw5KiUYL66NUee-MCzoE" '
                   '-O pretrained_models/SMIRK_em1.pt', shell=True, check=True)
print(f'Checkpoint: {CKPT.stat().st_size/1e6:.1f} MB')

# MediaPipe face-mesh model
MP_DIR = ROOT / 'mediapipe_models'
MP_DIR.mkdir(exist_ok=True)
mp_model = MP_DIR / 'face_landmarker_v2_with_blendshapes.task'
if not mp_model.exists():
    subprocess.run(
        'wget -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task '
        '-O mediapipe_models/face_landmarker_v2_with_blendshapes.task',
        shell=True, check=True)
print('MediaPipe model ready.')

# FLAME 2020 — user must upload FLAME2020.zip
FLAME_ZIP = pathlib.Path('/content/FLAME2020.zip')
ASSETS = ROOT / 'assets'
ASSETS.mkdir(exist_ok=True)
if not (ASSETS / 'FLAME2020').exists():
    if not FLAME_ZIP.exists():
        from google.colab import files
        print('Please upload your FLAME2020.zip (register at flame.is.tue.mpg.de):')
        uploaded = files.upload()
        FLAME_ZIP = pathlib.Path(list(uploaded.keys())[0])
    with zipfile.ZipFile(FLAME_ZIP) as z:
        z.extractall(ASSETS)
    # If zip extracted into a subdirectory, move it up
    sub = ASSETS / 'FLAME2020'
    if not sub.exists():
        for d in ASSETS.iterdir():
            if d.is_dir() and 'flame' in d.name.lower():
                d.rename(sub)
                break

print('FLAME2020 contents:', sorted(p.name for p in (ASSETS/'FLAME2020').iterdir()))

# landmark_embedding.npy is NOT in the official FLAME2020 zip —
# try several known sources
_lmk = ASSETS / 'FLAME2020' / 'landmark_embedding.npy'
if not _lmk.exists():
    print('Downloading landmark_embedding.npy ...')
    _lmk_urls = [
        # RingNet repo (soubhiksanyal)
        'https://github.com/soubhiksanyal/RingNet/raw/master/flame_model/landmark_embedding.npy',
        # FLAME_PyTorch repo (soubhiksanyal)
        'https://github.com/soubhiksanyal/FLAME_PyTorch/raw/master/FLAME_PyTorch/FLAME2020/landmark_embedding.npy',
        # DECA repo (yfeng1)
        'https://github.com/yfeng95/DECA/raw/master/data/landmark_embedding.npy',
    ]
    _downloaded = False
    for _url in _lmk_urls:
        _r = subprocess.run(f'wget -q "{_url}" -O "{_lmk}"', shell=True)
        if _r.returncode == 0 and _lmk.exists() and _lmk.stat().st_size > 1000:
            print(f'  ✔ Downloaded from {_url}')
            _downloaded = True
            break
        else:
            print(f'  ✗ Failed: {_url}')
            if _lmk.exists():
                _lmk.unlink()
    if not _downloaded:
        raise RuntimeError(
            'Could not download landmark_embedding.npy automatically.\n'
            'Please manually download it from one of these repos and upload to '
            f'{_lmk}:\n  ' + '\n  '.join(_lmk_urls)
        )
else:
    print('  ✔ landmark_embedding.npy already present.')


In [ ]:

# ── E4: Load SMIRK model components ─────────────────────────────────────
import sys, torch, pathlib, inspect, numpy as np, os
sys.path.insert(0, '/content/smirk')

# ── numpy 2.x back-compat aliases (FLAME/chumpy use removed names) ────────
if not hasattr(np, 'float_'):    np.float_    = np.float64
if not hasattr(np, 'int_'):      np.int_      = np.int64
if not hasattr(np, 'complex_'):  np.complex_  = np.complex128
if not hasattr(np, 'object_'):   np.object_   = object
if not hasattr(np, 'unicode_'):  np.unicode_  = np.str_
if not hasattr(np, 'string_'):   np.string_   = np.bytes_

# ── inspect patch — chumpy calls removed inspect.getargspec ───────────────
if not hasattr(inspect, 'getargspec'):
    inspect.getargspec = inspect.getfullargspec

from src.smirk_encoder import SmirkEncoder
from src.FLAME.FLAME import FLAME
from src.renderer.renderer import Renderer
from src.renderer.util import batch_orth_proj

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_PATH = '/content/smirk/pretrained_models/SMIRK_em1.pt'

# ── Verify FLAME assets ───────────────────────────────────────────────────
print('CWD:', os.getcwd())
print('assets/FLAME2020 contents:', sorted(os.listdir('assets/FLAME2020')))

_flame_model = 'assets/FLAME2020/generic_model.pkl'
_flame_lmk   = 'assets/FLAME2020/landmark_embedding.npy'
assert os.path.exists(_flame_model), f'Missing: {_flame_model} — re-run Cell 3'
assert os.path.exists(_flame_lmk),   f'Missing: {_flame_lmk} — re-run Cell 3'

# ── Encoder ───────────────────────────────────────────────────────────────
encoder = SmirkEncoder().to(DEVICE)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)

# The checkpoint wraps everything under a top-level model object.
# Determine the encoder sub-dict by trying common prefix patterns.
def _extract_encoder_sd(ckpt):
    if not isinstance(ckpt, dict):
        return ckpt
    # Try known top-level keys first
    for key in ('smirk_encoder', 'encoder'):
        if key in ckpt:
            return ckpt[key]
    # Try stripping a common prefix from flat keys
    # (checkpoint stores keys like "smirk_encoder.pose_encoder.xxx")
    for prefix in ('smirk_encoder.', 'encoder.'):
        filtered = {k[len(prefix):]: v for k, v in ckpt.items() if k.startswith(prefix)}
        if filtered:
            print(f'  Stripping prefix "{prefix}" → {len(filtered)} keys')
            return filtered
    # Fallback: return whole dict
    return ckpt

sd = _extract_encoder_sd(ckpt)
result = encoder.load_state_dict(sd, strict=False)
missing  = [k for k in result.missing_keys  if 'num_batches_tracked' not in k]
unexp    = result.unexpected_keys
print(f'Encoder loaded.  missing={len(missing)}  unexpected={len(unexp)}')
if missing:
    print(f'  First missing: {missing[:3]}')
encoder.eval()

# ── FLAME ─────────────────────────────────────────────────────────────────
_flame_sig = inspect.signature(FLAME.__init__).parameters
print(f'FLAME.__init__ params: {list(_flame_sig.keys())}')

_arg_map = {
    'flame_model_path':         _flame_model,
    'flame_lmk_embedding_path': _flame_lmk,
    'n_shape':                  300,
    'n_exp':                    50,
}
_flame_kwargs = {k: v for k, v in _arg_map.items() if k in _flame_sig}

flame = FLAME(**_flame_kwargs).to(DEVICE)
flame.eval()
print('FLAME loaded.')

# ── Renderer ──────────────────────────────────────────────────────────────
# Renderer.__init__(render_full_head=False, obj_filename='assets/head_template.obj')
# forward(vertices, cam_params, **landmarks) → {'rendered_img': (B,3,H,W), ...}
renderer = Renderer().to(DEVICE)

# Default camera: scale=8, no translation — used when encoder doesn't output cam
DEFAULT_CAM = torch.tensor([[8.0, 0.0, 0.0]], device=DEVICE)

def project_lmks(lmks3d, cam_params):
    """Project 3D landmarks → pixel coords (224×224) using orthographic projection."""
    proj = batch_orth_proj(lmks3d, cam_params)          # (B, N, 3)
    proj = proj[..., :2]
    proj[..., 1] = -proj[..., 1]                        # flip Y (renderer convention)
    px = (proj + 1.0) / 2.0 * 224.0                    # NDC [-1,1] → pixel [0,224]
    return px  # (B, N, 2)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f'SmirkEncoder : {count_params(encoder):,} trainable parameters')
print(f'FLAME        : {count_params(flame):,} parameters')
print(f'Device       : {DEVICE}')


---
## Section 1 — Landmark Alignment

We compute **Normalized Mean Error (NME)** between FLAME-projected 68 landmarks and MediaPipe pseudo-GT keypoints:

$$\text{NME} = \frac{1}{N} \sum_{i=1}^{N} \frac{\|\hat{\mathbf{l}}_i - \mathbf{l}_i\|_2}{d_{\text{io}}}$$

where $d_{\text{io}}$ is the inter-ocular distance (outer eye corners) and $N=68$.


In [ ]:

# ── E5: Helper functions — crop, encode, NME ─────────────────────────────
import cv2, mediapipe as mp, numpy as np, torch
from torchvision import transforms

IMG_SIZE = 224
TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

# MediaPipe face detector
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

MP_MODEL_PATH = '/content/smirk/mediapipe_models/face_landmarker_v2_with_blendshapes.task'
mp_opts = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MP_MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
)

def get_face_crop(bgr_img, target_size=IMG_SIZE):
    """Detect face via MediaPipe and return (cropped_rgb, lmks_norm, bbox)."""
    rgb = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    with FaceLandmarker.create_from_options(mp_opts) as fl:
        result = fl.detect(mp_img)
    if not result.face_landmarks:
        return None, None, None
    lmks = np.array([[lm.x, lm.y] for lm in result.face_landmarks[0]])
    H, W = rgb.shape[:2]
    lmks_px = lmks * [W, H]
    x1, y1 = lmks_px.min(axis=0)
    x2, y2 = lmks_px.max(axis=0)
    pad = 0.2 * max(x2 - x1, y2 - y1)
    x1, y1 = max(0, int(x1 - pad)), max(0, int(y1 - pad))
    x2, y2 = min(W, int(x2 + pad)), min(H, int(y2 + pad))
    crop = cv2.resize(rgb[y1:y2, x1:x2], (target_size, target_size))
    return crop, lmks, (x1, y1, x2, y2)

def encode_image(rgb_crop):
    """Run SMIRK encoder on a 224×224 RGB crop; returns full params dict."""
    tensor = TRANSFORM(rgb_crop).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = encoder(tensor)
    return out

def get_cam(params):
    """Extract cam_params from encoder output; fall back to a sensible default."""
    cam = params.get('cam', None)
    if cam is None:
        cam = DEFAULT_CAM
    return cam

def render_face(params, verts):
    """Render FLAME mesh; returns (rendered_rgb, mask) both in [0,1]."""
    cam = get_cam(params)
    with torch.no_grad():
        out = renderer(verts, cam)
    rendered = out['rendered_img'][0].clamp(0, 1)            # (3, H, W)
    mask     = (rendered.max(dim=0, keepdim=True).values     # derive mask from
                > 0.01).float()                              # non-black pixels
    return rendered, mask

def compute_nme(flame_lmks2d_px, mp_lmks_px):
    """NME with inter-ocular normalisation.
    flame_lmks2d_px: (68, 2) numpy array in pixel coords [0,224]
    mp_lmks_px     : (N, 2)  numpy array, first 68 used as pseudo-GT
    """
    pred = flame_lmks2d_px[:68]
    gt   = mp_lmks_px[:68]
    d_io = np.linalg.norm(gt[45] - gt[36]) + 1e-8   # outer eye corners (approx)
    nme  = np.mean(np.linalg.norm(pred - gt, axis=1)) / d_io
    return float(nme)

print('Helper functions defined.')


In [ ]:

# ── E6: Run NME on SMIRK sample images ───────────────────────────────────
import glob, pathlib, json, os
import pandas as pd

os.chdir('/content/smirk')   # ensure cwd is correct after any kernel restart
SAMPLE_DIR = pathlib.Path('/content/smirk/samples')
image_paths = sorted(glob.glob(str(SAMPLE_DIR / '*.jpg')) + glob.glob(str(SAMPLE_DIR / '*.png')))
print(f'Found {len(image_paths)} sample images.')

results = {}  # img_name -> {params, nme, ...}
nme_list = []

for path in image_paths:
    name = pathlib.Path(path).stem
    bgr = cv2.imread(path)
    if bgr is None:
        print(f'  SKIP {name}: cannot read'); continue

    crop, mp_lmks, bbox = get_face_crop(bgr)
    if crop is None:
        print(f'  SKIP {name}: no face detected'); continue

    params = encode_image(crop)
    cam    = get_cam(params)

    # Project FLAME landmarks to 2D pixel space
    with torch.no_grad():
        verts, lmks3d, _ = flame(
            shape_params=params.get('shape_params'),
            expression_params=params.get('expression_params'),
            pose_params=params.get('pose_params'),
        )
        lmks2d_px = project_lmks(lmks3d, cam)[0].cpu().numpy()  # (68, 2) pixels

    H, W = bgr.shape[:2]
    gt_scaled = (mp_lmks[:68] * [W, H] - [bbox[0], bbox[1]]) / max(bbox[2]-bbox[0], bbox[3]-bbox[1]) * 224

    nme = compute_nme(lmks2d_px, gt_scaled)
    nme_list.append(nme)

    results[name] = {
        'shape_norm': float(params['shape_params'].norm().cpu()),
        'exp_norm':   float(params['expression_params'].norm().cpu()),
        'jaw_angle':  float(params['pose_params'][0, 3].cpu()),
        'nme':        nme,
    }
    print(f'  {name:20s}  NME={nme:.4f}')

df = pd.DataFrame.from_dict(results, orient='index')
print(f'\nMean NME: {df.nme.mean():.4f} ± {df.nme.std():.4f}')
df.to_csv('/content/smirk_eval_nme.csv')
print('Saved: /content/smirk_eval_nme.csv')


---
## Section 2 — Reconstruction Fidelity (SSIM / PSNR / LPIPS)

We measure image-space fidelity between the **rendered FLAME mesh** and the **original crop** within the face foreground mask (alpha > 0.1) to avoid penalising background differences.

- **SSIM** (Structural Similarity Index) ↑ better
- **PSNR** (Peak Signal-to-Noise Ratio, dB) ↑ better
- **LPIPS** (Learned Perceptual Similarity, AlexNet) ↓ better


In [ ]:

# ── E7: SSIM / PSNR / LPIPS on face-masked region ────────────────────────
import torch, lpips, numpy as np
from skimage.metrics import structural_similarity as ssim_fn, peak_signal_noise_ratio as psnr_fn

loss_fn_lpips = lpips.LPIPS(net='alex').to(DEVICE)

fidelity_rows = []

for path in image_paths:
    name = pathlib.Path(path).stem
    if name not in results:
        continue

    bgr = cv2.imread(path)
    crop, _, _ = get_face_crop(bgr)
    if crop is None:
        continue

    params = encode_image(crop)

    with torch.no_grad():
        verts, lmks3d, _ = flame(
            shape_params=params.get('shape_params'),
            expression_params=params.get('expression_params'),
            pose_params=params.get('pose_params'),
        )

    rendered, mask = render_face(params, verts)   # (3,H,W), (1,H,W) in [0,1]

    gt_tensor = torch.from_numpy(crop).permute(2,0,1).float().to(DEVICE) / 255.0  # (3,H,W)

    # Apply foreground mask
    rendered_m = rendered * mask
    gt_m       = gt_tensor * mask

    # Convert to numpy HWC uint8 for skimage
    def t2np(t):
        return (t.permute(1,2,0).cpu().numpy() * 255).clip(0,255).astype(np.uint8)

    rnd_np = t2np(rendered_m); gt_np = t2np(gt_m)

    ssim_val  = ssim_fn(rnd_np, gt_np, channel_axis=2, data_range=255)
    psnr_val  = psnr_fn(gt_np, rnd_np, data_range=255)

    # LPIPS expects [-1,1] range
    def to_lpips(t): return (t * 2.0 - 1.0).unsqueeze(0)  # (1,3,H,W)
    with torch.no_grad():
        lpips_val = float(loss_fn_lpips(to_lpips(rendered_m), to_lpips(gt_m)))

    fidelity_rows.append({'name': name, 'ssim': ssim_val, 'psnr': psnr_val, 'lpips': lpips_val})
    print(f'  {name:20s}  SSIM={ssim_val:.4f}  PSNR={psnr_val:.2f}dB  LPIPS={lpips_val:.4f}')

df_fidelity = pd.DataFrame(fidelity_rows).set_index('name')
print(f'\nMean SSIM : {df_fidelity.ssim.mean():.4f} ± {df_fidelity.ssim.std():.4f}')
print(f'Mean PSNR : {df_fidelity.psnr.mean():.2f} ± {df_fidelity.psnr.std():.2f} dB')
print(f'Mean LPIPS: {df_fidelity.lpips.mean():.4f} ± {df_fidelity.lpips.std():.4f}')
df_fidelity.to_csv('/content/smirk_eval_fidelity.csv')


---
## Section 3 — Expression Space Analysis

SMIRK encodes expression as $\psi \in \mathbb{R}^{50}$ (FLAME expression PCA basis).  
We visualise the coefficient heatmap and characterise the dimensionality of the active expression subspace.


In [ ]:
# ── E8: Expression space — heatmap, active dims, PCA ─────────────────────
import matplotlib.pyplot as plt, seaborn as sns, numpy as np
from sklearn.decomposition import PCA

exp_matrix = []  # (N, 50)
img_names  = []

for path in image_paths:
    name = pathlib.Path(path).stem
    bgr = cv2.imread(path)
    if bgr is None: continue
    crop, _, _ = get_face_crop(bgr)
    if crop is None: continue
    params = encode_image(crop)
    exp_vec = params['expression_params'][0].cpu().numpy()  # (50,)
    exp_matrix.append(exp_vec)
    img_names.append(name)

exp_matrix = np.array(exp_matrix)  # (N, 50)
print(f'Expression matrix: {exp_matrix.shape}')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# (a) Heatmap
sns.heatmap(exp_matrix, ax=axes[0], cmap='RdBu_r', center=0,
            xticklabels=False, yticklabels=img_names,
            linewidths=0.3, linecolor='grey')
axes[0].set_title('Expression Coefficients ($\\psi \\in \\mathbb{R}^{50}$)')
axes[0].set_xlabel('Coefficient index'); axes[0].set_ylabel('Image')

# (b) Mean |ψ_i| per dimension
mean_abs = np.abs(exp_matrix).mean(axis=0)
thresh = mean_abs.mean()
active_dims = int((mean_abs > thresh).sum())
axes[1].bar(range(50), mean_abs, color=['#e74c3c' if v > thresh else '#3498db' for v in mean_abs])
axes[1].axhline(thresh, color='k', linestyle='--', linewidth=1)
axes[1].set_title(f'Mean |ψᵢ| per dim  (active={active_dims}/50)')
axes[1].set_xlabel('Coefficient index'); axes[1].set_ylabel('Mean |value|')

# (c) PCA explained variance
if exp_matrix.shape[0] >= 2:
    pca = PCA().fit(exp_matrix)
    axes[2].plot(np.cumsum(pca.explained_variance_ratio_) * 100, 'o-')
    axes[2].axhline(95, color='r', linestyle='--', linewidth=1, label='95%')
    axes[2].set_title('Cumulative PCA Explained Variance')
    axes[2].set_xlabel('# components'); axes[2].set_ylabel('Cumulative %')
    axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig('/content/smirk_expression_space.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Active expression dims (> mean |ψ|): {active_dims} / 50')
print(f'Mean  ||ψ||₂ across images: {np.linalg.norm(exp_matrix, axis=1).mean():.4f}')
print(f'Stdev ||ψ||₂ across images: {np.linalg.norm(exp_matrix, axis=1).std():.4f}')


---
## Section 4 — Shape Consistency & Disentanglement

A well-disentangled model should keep **shape** ($\beta$) stable across images of the same identity while letting **expression** ($\psi$) vary freely.  
We quantify:
- **Within-set shape variance** (lower → more consistent identity)
- **Pearson r(||ψ||₂, ||β||₂)** — should be near 0 for true disentanglement


In [ ]:
# ── E9: Shape consistency and disentanglement ─────────────────────────────
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr

shape_norms = []
exp_norms   = []
jaw_angles  = []
names_list  = []

for path in image_paths:
    name = pathlib.Path(path).stem
    bgr = cv2.imread(path)
    if bgr is None: continue
    crop, _, _ = get_face_crop(bgr)
    if crop is None: continue
    params = encode_image(crop)
    shape_norms.append(float(params['shape_params'].norm().cpu()))
    exp_norms.append(float(params['expression_params'].norm().cpu()))
    jaw_angles.append(float(params['pose_params'][0, 3].cpu()))
    names_list.append(name)

shape_norms = np.array(shape_norms)
exp_norms   = np.array(exp_norms)
jaw_angles  = np.array(jaw_angles)

var_shape = shape_norms.var()
r_val, p_val = pearsonr(exp_norms, shape_norms) if len(exp_norms) > 2 else (float('nan'), float('nan'))

print(f'Shape ||β||₂  mean={shape_norms.mean():.4f}  var={var_shape:.4f}')
print(f'Exp   ||ψ||₂  mean={exp_norms.mean():.4f}  var={exp_norms.var():.4f}')
print(f'Pearson r(exp_norm, shape_norm) = {r_val:.4f}  p={p_val:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x = range(len(names_list))
axes[0].bar(x, shape_norms, label='||β||₂ (shape)', alpha=0.7, color='#2ecc71')
axes[0].bar(x, exp_norms,   label='||ψ||₂ (expression)', alpha=0.7, color='#e74c3c')
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(names_list, rotation=45, ha='right')
axes[0].set_title('Parameter Norms per Image'); axes[0].legend()

axes[1].scatter(exp_norms, shape_norms, s=60, color='#3498db', edgecolors='k', linewidths=0.7)
for i, n in enumerate(names_list):
    axes[1].annotate(n, (exp_norms[i], shape_norms[i]), fontsize=7, ha='left', va='bottom')
axes[1].set_xlabel('||ψ||₂ (expression norm)')
axes[1].set_ylabel('||β||₂ (shape norm)')
axes[1].set_title(f'Disentanglement Scatter  r={r_val:.3f}')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/content/smirk_disentanglement.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Section 5 — Crop Ablation

SMIRK requires a tight face crop. We measure how much NME and SSIM degrade when the full uncropped image is passed directly vs. the median-filtered crop. Expected: 10–25% NME increase without crop.


In [ ]:

# ── E10: Crop vs no-crop ablation ────────────────────────────────────────
import cv2, torch, numpy as np
from skimage.metrics import structural_similarity as ssim_fn

delta_nme  = []
delta_ssim = []

for path in image_paths:
    name = pathlib.Path(path).stem
    if name not in results: continue

    bgr = cv2.imread(path)
    crop, mp_lmks, bbox = get_face_crop(bgr)
    if crop is None: continue

    # ── WITH crop (baseline) ──
    params_crop = encode_image(crop)
    cam_crop    = get_cam(params_crop)
    with torch.no_grad():
        verts, lmks3d, _ = flame(
            shape_params=params_crop['shape_params'],
            expression_params=params_crop['expression_params'],
            pose_params=params_crop['pose_params'],
        )
        lmks2d_crop_px = project_lmks(lmks3d, cam_crop)[0].cpu().numpy()

    H, W = bgr.shape[:2]
    gt_scaled = (mp_lmks[:68] * [W,H] - [bbox[0],bbox[1]]) / max(bbox[2]-bbox[0], bbox[3]-bbox[1]) * 224
    nme_crop = compute_nme(lmks2d_crop_px, gt_scaled)

    rendered_c, _ = render_face(params_crop, verts)
    rend_c = rendered_c.permute(1,2,0).cpu().numpy()
    gt_c   = crop / 255.0
    ssim_crop = ssim_fn(rend_c, gt_c, channel_axis=2, data_range=1.0)

    # ── WITHOUT crop (resize whole image to 224) ──
    full_resized = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), (224, 224))
    params_full = encode_image(full_resized)
    cam_full    = get_cam(params_full)
    with torch.no_grad():
        verts2, lmks3d2, _ = flame(
            shape_params=params_full['shape_params'],
            expression_params=params_full['expression_params'],
            pose_params=params_full['pose_params'],
        )
        lmks2d_full_px = project_lmks(lmks3d2, cam_full)[0].cpu().numpy()

    nme_full  = compute_nme(lmks2d_full_px, gt_scaled)
    rendered_f, _ = render_face(params_full, verts2)
    rend_f    = rendered_f.permute(1,2,0).cpu().numpy()
    ssim_full = ssim_fn(rend_f, full_resized/255.0, channel_axis=2, data_range=1.0)

    d_nme  = nme_full  - nme_crop
    d_ssim = ssim_crop - ssim_full
    delta_nme.append(d_nme);  delta_ssim.append(d_ssim)
    print(f'  {name:20s}  ΔNME={d_nme:+.4f}  ΔSSIM={d_ssim:+.4f}')

print(f'\nMean ΔNME  (full-crop): {np.mean(delta_nme):+.4f} ± {np.std(delta_nme):.4f}')
print(f'Mean ΔSSIM (crop-full): {np.mean(delta_ssim):+.4f} ± {np.std(delta_ssim):.4f}')


---
## Section 6 — Runtime Benchmark

Measures latency (ms), throughput (FPS), and peak VRAM for each module over 50 runs with 10-run warmup (batch size 1, CUDA events).


In [ ]:

# ── E11: Per-module latency benchmark (50 runs, CUDA events) ─────────────
import torch, gc, pandas as pd
import pathlib, cv2

# Use first available cropped image as benchmark input
bench_crop = None
for path in image_paths:
    bgr = cv2.imread(path)
    if bgr is None: continue
    c, _, _ = get_face_crop(bgr)
    if c is not None:
        bench_crop = c; break

assert bench_crop is not None, 'No valid image found for benchmarking.'

bench_tensor = TRANSFORM(bench_crop).unsqueeze(0).to(DEVICE)

def cuda_timer(fn, warmup=10, reps=50):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = torch.cuda.Event(enable_timing=True)
    t1 = torch.cuda.Event(enable_timing=True)
    t0.record()
    for _ in range(reps):
        fn()
    t1.record()
    torch.cuda.synchronize()
    ms = t0.elapsed_time(t1) / reps
    vram_mb = torch.cuda.max_memory_allocated() / 1e6
    return ms, vram_mb

rows = []

# Encoder
with torch.no_grad():
    ms, vram = cuda_timer(lambda: encoder(bench_tensor))
rows.append({'module': 'SmirkEncoder', 'latency_ms': ms, 'fps': 1000/ms, 'vram_mb': vram})

# Cache params for downstream
with torch.no_grad():
    p = encoder(bench_tensor)
bench_cam = get_cam(p)

# FLAME
with torch.no_grad():
    ms, vram = cuda_timer(lambda: flame(
        shape_params=p['shape_params'],
        expression_params=p['expression_params'],
        pose_params=p['pose_params'],
    ))
rows.append({'module': 'FLAME', 'latency_ms': ms, 'fps': 1000/ms, 'vram_mb': vram})

# Renderer
with torch.no_grad():
    verts, _, _ = flame(
        shape_params=p['shape_params'],
        expression_params=p['expression_params'],
        pose_params=p['pose_params'],
    )
    ms, vram = cuda_timer(lambda: renderer(verts, bench_cam))
rows.append({'module': 'Renderer', 'latency_ms': ms, 'fps': 1000/ms, 'vram_mb': vram})

# End-to-end
def e2e():
    with torch.no_grad():
        pp  = encoder(bench_tensor)
        c   = get_cam(pp)
        v, _, _ = flame(
            shape_params=pp['shape_params'],
            expression_params=pp['expression_params'],
            pose_params=pp['pose_params'],
        )
        renderer(v, c)
ms, vram = cuda_timer(e2e)
rows.append({'module': 'End-to-End', 'latency_ms': ms, 'fps': 1000/ms, 'vram_mb': vram})

df_perf = pd.DataFrame(rows).set_index('module')
df_perf = df_perf.round({'latency_ms': 2, 'fps': 1, 'vram_mb': 1})
print(df_perf.to_string())
df_perf.to_csv('/content/smirk_eval_runtime.csv')


---
## Section 7 — Resolution Robustness

We downsample the face crop to `[224, 128, 96, 64, 48, 32]` pixels before upscaling back to 224×224 and running inference. NME degradation quantifies reconstruction quality under low-resolution inputs.


In [ ]:

# ── E12: Resolution robustness curve ─────────────────────────────────────
import cv2, numpy as np, matplotlib.pyplot as plt

RESOLUTIONS = [224, 128, 96, 64, 48, 32]
res_nme = {r: [] for r in RESOLUTIONS}

for path in image_paths:
    name = pathlib.Path(path).stem
    bgr = cv2.imread(path)
    if bgr is None: continue
    crop, mp_lmks, bbox = get_face_crop(bgr)
    if crop is None: continue

    H, W = bgr.shape[:2]
    gt_scaled = (mp_lmks[:68] * [W,H] - [bbox[0],bbox[1]]) / max(bbox[2]-bbox[0], bbox[3]-bbox[1]) * 224

    for res in RESOLUTIONS:
        if res < 224:
            small = cv2.resize(crop, (res, res), interpolation=cv2.INTER_AREA)
            inp   = cv2.resize(small, (224, 224), interpolation=cv2.INTER_LINEAR)
        else:
            inp = crop
        params = encode_image(inp)
        cam    = get_cam(params)
        with torch.no_grad():
            _, lmks3d, _ = flame(
                shape_params=params['shape_params'],
                expression_params=params['expression_params'],
                pose_params=params['pose_params'],
            )
            lmks2d_px = project_lmks(lmks3d, cam)[0].cpu().numpy()
        nme = compute_nme(lmks2d_px, gt_scaled)
        res_nme[res].append(nme)

mean_nme = [np.mean(res_nme[r]) for r in RESOLUTIONS]
std_nme  = [np.std(res_nme[r])  for r in RESOLUTIONS]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.errorbar(RESOLUTIONS, mean_nme, yerr=std_nme, fmt='o-', capsize=5, color='#e74c3c', linewidth=2)
ax1.invert_xaxis()
ax1.set_xlabel('Input resolution (px)'); ax1.set_ylabel('NME')
ax1.set_title('NME vs Input Resolution'); ax1.grid(True)
ax1.set_xticks(RESOLUTIONS)

rel_nme = [m / mean_nme[0] for m in mean_nme]
ax2.plot(RESOLUTIONS, rel_nme, 's--', color='#3498db', linewidth=2)
ax2.axhline(1.0, color='k', linestyle=':', linewidth=1)
ax2.invert_xaxis()
ax2.set_xlabel('Input resolution (px)'); ax2.set_ylabel('Relative NME (224px = 1.0)')
ax2.set_title('Relative NME Degradation'); ax2.grid(True)
ax2.set_xticks(RESOLUTIONS)

plt.tight_layout()
plt.savefig('/content/smirk_resolution_robustness.png', dpi=150, bbox_inches='tight')
plt.show()

for res, m, s in zip(RESOLUTIONS, mean_nme, std_nme):
    print(f'  {res:3d}px → NME={m:.4f} ± {s:.4f}  ({m/mean_nme[0]:.2f}x baseline)')


---
## Section 8 — Qualitative Visualisation

Five-panel figure per image: **Input | FLAME render | Overlay | Expression bar | Pose bar**.
Saved as individual high-resolution PNGs and assembled into a publication-ready grid.


In [ ]:

# ── E13: Qualitative 5-panel figure ──────────────────────────────────────
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import numpy as np, torch, cv2, pathlib

EXP_NAMES = [f'exp_{i:02d}' for i in range(50)]
POSE_NAMES = ['rot_x', 'rot_y', 'rot_z', 'jaw_x', 'jaw_y', 'jaw_z']
MAX_FIGURES = 5  # limit for Colab memory

for idx, path in enumerate(image_paths[:MAX_FIGURES]):
    name = pathlib.Path(path).stem
    bgr = cv2.imread(path)
    if bgr is None: continue
    crop, _, _ = get_face_crop(bgr)
    if crop is None: continue

    params = encode_image(crop)
    with torch.no_grad():
        verts, lmks3d, _ = flame(
            shape_params=params['shape_params'],
            expression_params=params['expression_params'],
            pose_params=params['pose_params'],
        )

    rendered, mask = render_face(params, verts)   # (3,H,W), (1,H,W)
    rendered_np = rendered.permute(1,2,0).cpu().numpy()           # HWC [0,1]
    mask_np     = mask[0].cpu().numpy()                           # HW  {0,1}
    overlay     = np.where(mask_np[..., None] > 0.5,
                           rendered_np * 0.6 + (crop/255.0) * 0.4,
                           crop/255.0)

    exp_vals  = params['expression_params'][0].cpu().numpy()
    pose_vals = params['pose_params'][0].cpu().numpy()

    fig = plt.figure(figsize=(22, 5))
    fig.suptitle(f'SMIRK Reconstruction: {name}', fontsize=14, fontweight='bold')

    ax1 = fig.add_subplot(1, 5, 1)
    ax1.imshow(crop); ax1.set_title('Input (cropped)'); ax1.axis('off')

    ax2 = fig.add_subplot(1, 5, 2)
    ax2.imshow(rendered_np); ax2.set_title('FLAME Render'); ax2.axis('off')

    ax3 = fig.add_subplot(1, 5, 3)
    ax3.imshow(np.clip(overlay, 0, 1)); ax3.set_title('Overlay (α=0.6)'); ax3.axis('off')

    ax4 = fig.add_subplot(1, 5, 4)
    cols4 = ['#e74c3c' if v > 0 else '#3498db' for v in exp_vals[:20]]
    ax4.barh(range(20), exp_vals[:20], color=cols4)
    ax4.set_yticks(range(20)); ax4.set_yticklabels([f'ψ_{i}' for i in range(20)], fontsize=7)
    ax4.axvline(0, color='k', linewidth=0.8)
    ax4.set_title('Top-20 Exp Coeffs'); ax4.set_xlabel('Value')

    ax5 = fig.add_subplot(1, 5, 5)
    cols5 = ['#e74c3c' if v > 0 else '#3498db' for v in pose_vals]
    ax5.bar(POSE_NAMES, pose_vals, color=cols5)
    ax5.axhline(0, color='k', linewidth=0.8)
    ax5.set_title('Pose Parameters (θ)'); ax5.set_ylabel('Radians')
    plt.setp(ax5.get_xticklabels(), rotation=30, ha='right', fontsize=8)

    plt.tight_layout()
    plt.savefig(f'/content/smirk_qual_{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: /content/smirk_qual_{name}.png')


---
## Section 9 — Statistical Summary & LaTeX Table

Aggregated mean ± std across all evaluated images for every metric. Automatically generates a `\\table{}` suitable for direct inclusion in a LaTeX paper.


In [ ]:
# ── E14: Merge results, compute mean±std, generate LaTeX table ───────────
import pandas as pd, numpy as np

# Merge all per-image metrics
df_all = df.copy()  # NME from E6
if 'df_fidelity' in dir():
    df_all = df_all.join(df_fidelity, how='outer')

print('=== Per-Image Summary ===')
print(df_all.round(4).to_string())
print()

summary = df_all.agg(['mean', 'std']).T
summary.columns = ['Mean', 'Std']
summary['Mean±Std'] = summary.apply(lambda r: f'{r.Mean:.4f} ± {r.Std:.4f}', axis=1)

print('=== Aggregate Summary ===')
print(summary[['Mean±Std']].to_string())
print()

# ── LaTeX table ──
latex_lines = [
    r'\begin{table}[htbp]',
    r'  \centering',
    r'  \caption{SMIRK Evaluation Results}',
    r'  \label{tab:smirk_eval}',
    r'  \begin{tabular}{lcc}',
    r'    \hline',
    r'    \textbf{Metric} & \textbf{Mean} & \textbf{Std} \\',
    r'    \hline',
]

metric_display = {'nme': 'NME ↓', 'ssim': 'SSIM ↑', 'psnr': 'PSNR (dB) ↑', 'lpips': 'LPIPS ↓',
                  'shape_norm': '$\\|\\beta\\|_2$', 'exp_norm': '$\\|\\psi\\|_2$',
                  'jaw_angle': 'Jaw angle (rad)'}

for col in df_all.columns:
    disp = metric_display.get(col, col)
    m = df_all[col].mean(); s = df_all[col].std()
    latex_lines.append(f'    {disp} & {m:.4f} & {s:.4f} \\\\')

latex_lines += [
    r'    \hline',
    r'  \end{tabular}',
    r'\end{table}',
]

latex_src = '\n'.join(latex_lines)
print('=== LaTeX Table ===')
print(latex_src)

with open('/content/smirk_eval_table.tex', 'w') as f:
    f.write(latex_src)
print('\nSaved: /content/smirk_eval_table.tex')

df_all.round(4).to_csv('/content/smirk_eval_full.csv')
print('Saved: /content/smirk_eval_full.csv')


In [ ]:
# ── E15: Bundle all artefacts and download ────────────────────────────────
import tarfile, glob, pathlib, os
from google.colab import files

ARTEFACTS = [
    '/content/smirk_eval_nme.csv',
    '/content/smirk_eval_fidelity.csv',
    '/content/smirk_eval_runtime.csv',
    '/content/smirk_eval_full.csv',
    '/content/smirk_eval_table.tex',
    '/content/smirk_expression_space.png',
    '/content/smirk_disentanglement.png',
    '/content/smirk_resolution_robustness.png',
]

# Also add qualitative figures
ARTEFACTS += glob.glob('/content/smirk_qual_*.png')

TAR_PATH = '/content/smirk_evaluation_results.tar.gz'
with tarfile.open(TAR_PATH, 'w:gz') as tar:
    for f in ARTEFACTS:
        if pathlib.Path(f).exists():
            tar.add(f, arcname=pathlib.Path(f).name)
            print(f'  + {pathlib.Path(f).name}')
        else:
            print(f'  ! MISSING: {f}')

size_mb = pathlib.Path(TAR_PATH).stat().st_size / 1e6
print(f'\nArchive: {TAR_PATH}  ({size_mb:.2f} MB)')
files.download(TAR_PATH)
